# Lösung 2: Generative Adversarial Networks

In dieser Aufgabe wird ein DCGAN auf Basis des [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) Datensatzes trainiert.

# 1. Nötige Imports durchnehmen

Die folgende Code-Zelle importiert die notwendigen Bibliotheken, die im Folgenden benötigt werden.



In [ ]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
from torch.autograd import Variable
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from tqdm.notebook import tqdm

# 2. Trainingsdaten laden

Zunächst werden die Trainingsdaten aus dem Internet geladen:

Dabei verwenden wir den [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) Datensatz, welcher Bilder von Flugzeugen, Autos, Vögeln etc. beinhaltet. Die Daten werden im Verzeichnis ``/tmp/data`` gespeichert.

Als alternativer Datensatz kann auch das [102 Category Flower Dataset](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/) verwendet werden.

Anschließend wird eine Bildtransformation definiert, bei der die Bilder zunächst auf eine Größe von $64\times 64$ Pixel skaliert werden. Danach werden die Bilder in Tensoren umgewandelt und anschließend normalisiert, sodass die Werte der Pixel in einem Bereich von $-1$ bis $1$ liegen.

Der Datensatz wird danach mithilfe des DataLoader in handliche Batches aufgeteilt. Gleichzeitig wird das Shuffling der Daten durch den Parameter `shuffle=True` aktiviert, damit das Modell während des Trainings möglichst vielfältige Beispiele erhält. Außerdem wird das parallele Laden von Daten über mehrere (hier $5$) Prozesse ermöglicht (Paramater `num_workers=5`), um die Effizienz zu steigern.

Zuletzt wird entschieden, auf welchem Gerät das Training stattfinden soll. Wenn eine (NVIDIA) GPU verfügbar ist, wird diese genutzt, andernfalls erfolgt das Training auf der CPU.

In [ ]:
img_size = 64
#batch_size=64
batch_size = 256
lr = 0.0002
beta1 = 0.5
num_epochs = 25
outf= 'output'
dataset = 'CIFAR10'
#dataset = 'Flowers102'

if dataset == 'CIFAR10':
    # download CIFAR10 data 
    dataset = datasets.CIFAR10(root = '/tmp/data',download=True,
                               transform=transforms.Compose([
                               transforms.Resize(img_size),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                           ]))

elif dataset == 'Flowers102':
    # as an alternative dataset you can use the Flowers102 dataset 
    dataset = datasets.Flowers102(root="/tmp/data", download = True, 
                                   transform=transforms.Compose([
                                   transforms.Resize(img_size),
                                   transforms.CenterCrop(img_size),
                                   transforms.ToTensor(),
                                   transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                               ]))
else:
    print('Wrong dataset chosen?')


dataloader = torch.utils.data.DataLoader(dataset, batch_size, num_workers=5, shuffle=True)

# Decide which device we want to run on
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Hyperparameter

In der nächsten Zelle werden Hyperparameter gewählt wie die Dimension des latenten Raums oder die Zahl der Features für Generator und Diskriminator.

Bei GANs ist es ungünstig, wenn der Diskriminator deutlich besser ist als der Generator. Die Zahl der Features bzw. das Verhältnis zwischen Generator und Diskriminator kann hier einen Einfluss haben.


In [ ]:
#Size of latnet vector
nz = 100
# Filter size of generator
ngf = 64
# Filter size of discriminator
ndf = 64
# Output image channels
nc = 3

# Initialisierung der Gewichte

Bei GANs ist die Initialisierung der Gewichte in den Netzwerken wichtig. Die folgende Funktion basiert auf Best Practices.

Die Funktion überprüft, ob es sich bei dem übergebenen Modul um eine Convolution-Schicht oder eine Batch-Normalisierung handelt. Für Convolution-Schichten werden die Gewichte mit Werten aus einer Normalverteilung mit Mittelwert $0$ und Standardabweichung $0.02$ initialisiert. Bei BatchNorm-Schichten werden die Gewichte ebenfalls normalverteilt initialisiert, allerdings mit einem Mittelwert von $1$. Zusätzlich wird der Bias dieser Schichten auf $0$ gesetzt. Diese Form der Initialisierung hilft, Probleme wie das Verschwinden oder Explodieren von Gradienten zu vermeiden und verbessert die Konvergenz während des Trainings.



In [ ]:
def weights_inititialisation(m):
    class_name = m.__class__.__name__
    if class_name.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif class_name.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)

# Generator-Netzwerk

In dieser Zelle wird das Generator-Netzwerk für das GAN definiert. Der Generator hat die Aufgabe, aus einem zufälligen Rauschvektor (dem latenten Vektor) realistisch wirkende Bilder zu erzeugen.



Das Netzwerk basiert auf einer Abfolge von [ConvTranspose2d](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)-Schichten, also transponierten Convolution-Layern, die eine schrittweise Vergrößerung der Bilddimension ermöglichen. Zwischen diesen Schichten werden [BatchNorm2d](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html) und [ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)-Aktivierungsfunktionen eingesetzt, um die Trainingsstabilität zu erhöhen und nicht-lineare Abbildungen zu ermöglichen.

Beginnend mit einem Eingabe-Tensor der Form ``nz`` wird die Bildgröße schrittweise durch die Transposed Convolutions auf die endgültige Zielgröße gebracht – in diesem Fall ``64×64`` Pixel. Die letzte Schicht erzeugt ein RGB-Bild ``nc = 3`` und verwendet eine [Tanh](https://pytorch.org/docs/stable/generated/torch.nn.Tanh.html)-Aktivierungsfunktion, um die Ausgabewerte in den Bereich ``[-1, 1]`` zu bringen, was zur vorher definierten Normalisierung der Eingabedaten passt.

Nach der Definition des Netzwerks wird es mithilfe der zuvor beschriebenen Initialisierungsfunktion mit geeigneten Startwerten versehen.



In [ ]:
class _net_generator(nn.Module):
    def __init__(self):
        super(_net_generator, self).__init__()

        self.main = nn.Sequential(
            nn.ConvTranspose2d(     nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2,     ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(    ngf,      nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        output = self.main(input)
        return output


net_generator = _net_generator()
net_generator.apply(weights_inititialisation)

print('Generator infos: ')
print(net_generator)

# Diskriminator-Netzwerk

In dieser Zelle wird der Diskriminator des GANs definiert. Der Diskriminator ist ein Convolutional Neural Network (CNN), das ein Bild als Eingabe erhält und entscheidet, ob es sich um ein echtes Bild aus dem Trainingsdatensatz oder ein vom Generator erzeugtes Bild handelt.

Das Netzwerk besteht aus mehreren Conv2d-Schichten, die die Bilddimensionen schrittweise verkleinern und gleichzeitig die Anzahl der Feature-Maps erhöhen. Nach jeder Convolution folgt eine [LeakyReLU](https://pytorch.org/docs/stable/generated/torch.nn.LeakyReLU.html)-Aktivierungsfunktion, die auch bei negativen Eingaben einen geringen Gradienten ermöglicht und damit das "Absterben" von Neuronen verhindert. Ab der zweiten Schicht wird zusätzlich BatchNorm2d verwendet, um das Training zu stabilisieren.

Am Ende des Netzwerks wird ein einzelner Wert pro Bild berechnet, der mittels einer [Sigmoid](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html)-Aktivierungsfunktion in den Bereich ``[0, 1]`` gebracht wird. Dieser Wert entspricht der geschätzten Wahrscheinlichkeit, dass das eingegebene Bild „echt“ ist.

Wie bereits beim Generator wird auch hier die Initialisierung der Gewichte mit der vorher definierten Methode durchgeführt, um ein stabiles Training zu unterstützen.



In [ ]:
class _net_discriminator(nn.Module):
    def __init__(self):
        super(_net_discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        output = self.main(input)
        return output.view(-1, 1).squeeze(1)


net_discriminator = _net_discriminator()
net_discriminator.apply(weights_inititialisation)
print('Discriminator infos: ')
print(net_discriminator)

# Definition der Verlustfunktion und einiger Hilfswerte

In dieser Zelle wird zunächst die Verlustfunktion für das GAN definiert. Es wird die [BCELoss](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html) (Binary Cross Entropy Loss) verwendet, die sich gut für binäre Klassifikationsprobleme eignet. Im Fall von GANs bewertet sie, wie gut der Diskriminator zwischen echten und generierten Bildern unterscheiden kann.

Zusätzlich werden einige Hilfs-Tensoren erstellt, die während des Trainings verwendet werden. Der Tensor input dient als Platzhalter für reale Bilder aus dem Datensatz. Der Tensor ``noise`` wird als Eingabe für den Generator verwendet und enthält zufällige Rauschvektoren. ``fixed_noise`` ist ein spezieller, einmalig generierter Rauschvektor, der während des Trainings konstant bleibt und zur qualitativen Beurteilung der Generatorentwicklung genutzt wird.

Der Tensor ``label`` speichert die Zielwerte (Labels) für den Diskriminator – entweder `real_label` (`1`) für echte Bilder oder `fake_label` (`0`) für generierte Bilder. Diese Werte werden im Trainingsprozess verwendet, um festzulegen, was als „echt“ oder „gefälscht“ gelten soll.



In [ ]:
criterion = nn.BCELoss()

input = torch.FloatTensor(batch_size, 3, img_size, img_size)
noise = torch.FloatTensor(batch_size, nz, 1, 1)
fixed_noise = torch.FloatTensor(batch_size, nz, 1, 1).normal_(0, 1)
label = torch.FloatTensor(batch_size)
real_label = 1
fake_label = 0

### Verwendung der GPU (CUDA)

Check ob eine CUDA GPU zur Verfügung steht

In [ ]:
if torch.cuda.is_available():
    net_discriminator.cuda()
    net_generator.cuda()
    criterion.cuda()
    input, label = input.cuda(), label.cuda()
    noise, fixed_noise = noise.cuda(), fixed_noise.cuda()

# Definition des Optimizers

In dieser Zelle werden die Optimierungsverfahren für den Generator und den Diskriminator festgelegt. Dabei kommt der [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)-Optimizer zum Einsatz, der sich durch eine adaptive Lernratenanpassung auszeichnet und häufig bei GANs verwendet wird

In [ ]:
fixed_noise = Variable(fixed_noise)

optimizer_discriminator = optim.Adam(net_discriminator.parameters(), lr, betas=(beta1, 0.95))
optimizer_generator = optim.Adam(net_generator.parameters(), lr, betas=(beta1, 0.95))

# Trainingsschleife 

In jeder Epoche wird zunächst der Diskriminator trainiert. Dabei verarbeitet er pro Schritt zwei Bildbatches: einen mit echten Bildern aus dem Datensatz und einen mit künstlich erzeugten Bildern vom Generator. Für beide berechnet er jeweils eine Vorhersage und erhält entsprechend den Verlust, der durch den Vergleich mit den zugewiesenen Labeln ("echt" bzw. "fake") entsteht. Ziel des Diskriminators ist es, diese beiden Bildtypen korrekt zu unterscheiden.

Im Anschluss wird der Generator optimiert. Sein Ziel ist es, den Diskriminator möglichst zu täuschen – das heißt, Bilder zu erzeugen, die so realistisch wirken, dass der Diskriminator sie für echte Bilder hält. Deshalb wird sein Verlust gegen das Label "real" berechnet, obwohl die erzeugten Bilder künstlich sind. Ein niedriger Generator-Verlust bedeutet somit, dass die generierten Bilder überzeugend genug sind, um den Diskriminator zu verwirren.

Zusätzlich werden während des Trainings verschiedene Metriken (Verluste, Diskriminator-Ausgaben) zur späteren Analyse gespeichert. Außerdem wird in regelmäßigen Abständen ein fester Satz an Rauschvektoren durch den Generator geleitet, um dessen Fortschritt anhand gleichbleibender Eingaben verfolgen zu können.



In [ ]:
# Training Loop

# Lists to keep track of progress
img_list = []
G_losses = []
D_losses = []
iters = 0

print("Starting Training Loop...")
# For each epoch

with tqdm(range(num_epochs)) as pbar: 
    for epoch in pbar:
        # For each batch in the dataloader
        for i, (data, label) in enumerate(tqdm(dataloader), 0):

            ############################
            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            ###########################
            ## Train with all-real batch
            net_discriminator.zero_grad()
            # Format batch
            real_cpu = data.to(device)
            b_size = real_cpu.size(0)
            label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
            # Forward pass real batch through D
            output = net_discriminator(real_cpu).view(-1)
            # Calculate loss on all-real batch
            errD_real = criterion(output, label)
            # Calculate gradients for D in backward pass
            errD_real.backward()
            D_x = output.mean().item()

            ## Train with all-fake batch
            # Generate batch of latent vectors
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            # Generate fake image batch with G
            fake = net_generator(noise)
            label.fill_(fake_label)
            # Classify all fake batch with D
            output = net_discriminator(fake.detach()).view(-1)
            # Calculate D's loss on the all-fake batch
            errD_fake = criterion(output, label)
            # Calculate the gradients for this batch, accumulated (summed) with previous gradients
            errD_fake.backward()
            D_G_z1 = output.mean().item()
            # Compute error of D as sum over the fake and the real batches
            errD = errD_real + errD_fake
            # Update D
            optimizer_discriminator.step()

            ############################
            # (2) Update G network: maximize log(D(G(z)))
            ###########################
            net_generator.zero_grad()
            label.fill_(real_label)  # fake labels are real for generator cost
            # Since we just updated D, perform another forward pass of all-fake batch through D
            output = net_discriminator(fake).view(-1)
            # Calculate G's loss based on this output
            errG = criterion(output, label)
            # Calculate gradients for G
            errG.backward()
            D_G_z2 = output.mean().item()
            # Update G
            optimizer_generator.step()

            # Output training stats
            if i % 50 == 0:
                pbar.set_postfix({ "Loss_D": errD.item(), "Loss_G": errG.item(), "D(x)": D_x, "D(G(z1))": D_G_z1, "D(G(z2))": D_G_z2})
            #    print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
            #          % (epoch, num_epochs, i, len(dataloader),
            #             errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

            # Save Losses for plotting later
            G_losses.append(errG.item())
            D_losses.append(errD.item())

            # Check how the generator is doing by saving G's output on fixed_noise
            if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
                with torch.no_grad():
                    fake = net_generator(fixed_noise).detach().cpu()
                img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

            iters += 1

## Visualisierung der Trainingsverluste

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses,label="G")
plt.plot(D_losses,label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Animation der generierten Bilder

In diesem Abschnitt wird eine Animation erstellt, die den Fortschritt des Generators während des Trainings zeigt. Die Bilder stammen aus dem regelmäßig gespeicherten img_list, das Bilder zeigt, die aus einem konstanten Rauschvektor ``(fixed_noise)`` erzeugt wurden.

Durch die Darstellung als zeitliche Abfolge kann nachvollzogen werden, wie sich die Qualität und Struktur der generierten Bilder mit fortschreitendem Training verändert. Dies ist besonders hilfreich, um die Lernentwicklung des Generators visuell zu evaluieren.

**Runterscrollen** um die Animation abzuspielen



In [ ]:
fig = plt.figure(figsize=(8,8))
plt.axis("off")
ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated=True)] for i in img_list]
ani = animation.ArtistAnimation(fig, ims, interval=1000, repeat_delay=1000, blit=True)

HTML(ani.to_jshtml())